In [ ]:
# Save processed data for backend API
# Save the main dataset
data_path = r'c:\Users\Asus\OneDrive\Documents\Campus Pulse Student Satisfaction Dashboard\data\student_satisfaction_data.csv'
df.to_csv(data_path, index=False)
print(f"\n✓ Main dataset saved to: {data_path}")

# Save key metrics as JSON
import json

metrics_dict = {
    'overall_metrics': {
        'average_score': float(overall_avg),
        'std_dev': float(overall_std),
        'total_responses': int(len(df))
    },
    'facility_metrics': facility_metrics.to_dict(),
    'best_facility': best_facility,
    'worst_facility': worst_facility,
    'academic_year_metrics': year_metrics.to_dict()
}

metrics_path = r'c:\Users\Asus\OneDrive\Documents\Campus Pulse Student Satisfaction Dashboard\data\key_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_dict, f, indent=4, default=str)
print(f"✓ Key metrics saved to: {metrics_path}")

print("\n" + "=" * 70)
print("✓ DATA ANALYSIS COMPLETE - Ready for backend integration!")
print("=" * 70)

In [ ]:
# Calculate Key Metrics

print("=" * 70)
print("KEY METRICS FOR CAMPUS PULSE DASHBOARD")
print("=" * 70)

# 1. Overall Average Satisfaction Score
overall_avg = df['satisfaction_score'].mean()
overall_std = df['satisfaction_score'].std()
print(f"\n1. OVERALL AVERAGE SATISFACTION SCORE")
print(f"   Average: {overall_avg:.2f} / 5.0")
print(f"   Std Dev: {overall_std:.2f}")
print(f"   Total Responses: {len(df)}")

# 2. Satisfaction by Facility
print(f"\n2. AVERAGE SATISFACTION BY FACILITY")
facility_metrics = df.groupby('facility_rated')['satisfaction_score'].agg(['mean', 'count', 'std']).sort_values('mean', ascending=False)
facility_metrics.columns = ['avg_score', 'response_count', 'std_dev']
print(facility_metrics.to_string())

# 3. Best and Worst Facilities
best_facility = facility_metrics['avg_score'].idxmax()
worst_facility = facility_metrics['avg_score'].idxmin()
print(f"\n3. BEST AND WORST PERFORMING FACILITIES")
print(f"   Best Facility: {best_facility} (Avg: {facility_metrics.loc[best_facility, 'avg_score']:.2f})")
print(f"   Worst Facility: {worst_facility} (Avg: {facility_metrics.loc[worst_facility, 'avg_score']:.2f})")

# 4. Satisfaction by Academic Year
print(f"\n4. AVERAGE SATISFACTION BY ACADEMIC YEAR")
year_metrics = df.groupby('academic_year')['satisfaction_score'].agg(['mean', 'count', 'std'])
year_metrics.columns = ['avg_score', 'response_count', 'std_dev']
print(year_metrics.to_string())

# 5. Satisfaction by Major
print(f"\n5. TOP 5 MAJORS BY AVERAGE SATISFACTION")
major_metrics = df.groupby('major')['satisfaction_score'].agg(['mean', 'count']).sort_values('mean', ascending=False)
major_metrics.columns = ['avg_score', 'response_count']
print(major_metrics.head(5).to_string())

# 6. Satisfaction Category Distribution
print(f"\n6. SATISFACTION CATEGORY DISTRIBUTION")
category_dist = df['satisfaction_category'].value_counts()
category_pct = (category_dist / len(df) * 100).round(2)
for cat in ['Very Unsatisfied', 'Unsatisfied', 'Neutral', 'Satisfied', 'Very Satisfied']:
    if cat in category_dist.index:
        print(f"   {cat:20s}: {category_dist[cat]:4d} ({category_pct[cat]:5.2f}%)")

## Section 6: Developing Key Metrics

Calculate and save key metrics for the dashboard backend.

In [ ]:
# 4. Heatmap: Satisfaction by Facility and Academic Year
pivot_data = df.pivot_table(values='satisfaction_score', index='facility_rated', columns='academic_year', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_data, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Avg Satisfaction'}, vmin=1, vmax=5)
ax.set_title('Average Satisfaction Score: Facility vs Academic Year')
plt.tight_layout()
plt.show()

print("✓ Heatmap visualization created")

In [ ]:
# 3. Satisfaction Score by Academic Year
academic_scores = df.groupby('academic_year')['satisfaction_score'].agg(['mean', 'std', 'count'])

fig, ax = plt.subplots(figsize=(10, 6))
academic_scores['mean'].plot(kind='bar', ax=ax, color='lightblue', edgecolor='black', alpha=0.7, yerr=academic_scores['std'], capsize=5)
ax.set_xlabel('Academic Year')
ax.set_ylabel('Average Satisfaction Score')
ax.set_title('Average Satisfaction Score by Academic Year')
ax.set_ylim(0, 5)
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Academic year analysis visualization created")
print(f"\nAcademic Year Statistics:")
print(academic_scores)

In [ ]:
# 2. Average Satisfaction Score by Facility
facility_scores = df.groupby('facility_rated')['satisfaction_score'].agg(['mean', 'count']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
facility_scores['mean'].plot(kind='barh', color='lightgreen', edgecolor='black', alpha=0.7, ax=ax)
ax.set_xlabel('Average Satisfaction Score')
ax.set_ylabel('Facility')
ax.set_title('Average Satisfaction Score by Facility')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(facility_scores['mean']):
    ax.text(v + 0.05, i, f'{v:.2f}', va='center')

plt.tight_layout()
plt.show()

print("✓ Facility-wise satisfaction visualization created")
print(f"\nFacility Statistics:")
print(facility_scores)

In [ ]:
# 1. Distribution of Satisfaction Scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['satisfaction_score'], bins=5, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Satisfaction Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Satisfaction Scores')
axes[0].grid(axis='y', alpha=0.3)

# Bar plot for satisfaction categories
satisfaction_counts = df['satisfaction_category'].value_counts().sort_index()
satisfaction_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Satisfaction Category')
axes[1].set_ylabel('Count')
axes[1].set_title('Satisfaction by Category')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Distribution visualizations created")

## Section 5: Exploratory Data Analysis (EDA)

Visualize data distribution, patterns, and relationships.

In [ ]:
# Create new features for better analysis

# 1. Categorize satisfaction scores
def categorize_score(score):
    if score <= 1:
        return 'Very Unsatisfied'
    elif score == 2:
        return 'Unsatisfied'
    elif score == 3:
        return 'Neutral'
    elif score == 4:
        return 'Satisfied'
    else:
        return 'Very Satisfied'

df['satisfaction_category'] = df['satisfaction_score'].apply(categorize_score)

# 2. Extract month and year for time-based analysis
df['survey_month'] = pd.to_datetime(df['timestamp']).dt.to_period('M')
df['survey_year'] = pd.to_datetime(df['timestamp']).dt.year

# 3. Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("✓ Data preprocessing completed!")
print(f"\nNew features created:")
print(f"  - satisfaction_category: {df['satisfaction_category'].unique()}")
print(f"  - survey_month: Time period for trend analysis")
print(f"  - survey_year: Year extraction")
print(f"\nUpdated dataset shape: {df.shape}")
print(f"\nSample of processed data:")
print(df.head())

## Section 4: Data Cleaning and Preprocessing

Create features and prepare data for analysis and visualization.

In [ ]:
# Data Shape and Info
print("=" * 60)
print("DATASET SHAPE AND INFORMATION")
print("=" * 60)
print(f"Dataset shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

# Check for missing values
print("\n" + "=" * 60)
print("MISSING VALUES CHECK")
print("=" * 60)
missing = df.isnull().sum()
if missing.sum() == 0:
    print("✓ No missing values found!")
else:
    print(f"Missing values:\n{missing}")

# Basic descriptive statistics
print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
print(df.describe())

# Categorical columns overview
print("\n" + "=" * 60)
print("CATEGORICAL COLUMNS OVERVIEW")
print("=" * 60)
print(f"Unique Academic Years: {df['academic_year'].nunique()} - {sorted(df['academic_year'].unique())}")
print(f"Unique Majors: {df['major'].nunique()}")
print(f"Unique Facilities: {df['facility_rated'].nunique()} - {sorted(df['facility_rated'].unique())}")

## Section 3: Initial Data Exploration

Check data structure, missing values, data types, and basic statistics.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Parameters for data generation
n_records = 500
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31)

# Define choices
academic_years = ['2023-2024', '2024-2025']
majors = ['Computer Science', 'Engineering', 'Business', 'Arts', 'Medicine', 
          'Law', 'Education', 'Sciences']
facilities = ['Library', 'Cafeteria', 'Sports Center', 'Dormitory', 
              'Student Center', 'Lab Facilities', 'Parking', 'Medical Center']

# Generate synthetic data
np.random.seed(42)
data = {
    'student_id': [f'STU{str(i).zfill(5)}' for i in range(1, n_records + 1)],
    'academic_year': np.random.choice(academic_years, n_records),
    'major': np.random.choice(majors, n_records),
    'facility_rated': np.random.choice(facilities, n_records),
    'satisfaction_score': np.random.choice([1, 2, 3, 4, 5], n_records, p=[0.05, 0.10, 0.20, 0.35, 0.30]),
    'timestamp': [start_date + timedelta(days=np.random.randint(0, (end_date - start_date).days)) 
                  for _ in range(n_records)],
    'feedback': np.random.choice([
        'Excellent service', 'Good experience', 'Average facilities',
        'Needs improvement', 'Poor quality', 'Satisfactory', 'Very satisfied', 'Not satisfied'
    ], n_records)
}

# Create DataFrame
df = pd.DataFrame(data)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"✓ Synthetic dataset generated successfully!")
print(f"✓ Total records: {len(df)}")
print(f"\nFirst 10 rows of the dataset:")
print(df.head(10))

## Section 2: Data Acquisition - Creating Synthetic Dataset

Generate a synthetic dataset with 500+ student satisfaction survey responses.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visualization styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully!")

KeyboardInterrupt: 

## Section 1: Project Setup and Tool Installation

Install and import required libraries for data analysis and visualization.

# Campus Pulse - Student Satisfaction Dashboard
## Exploratory Data Analysis and Metric Development

This notebook covers:
1. Project Setup and Tool Installation
2. Data Acquisition (Synthetic Dataset Generation)
3. Initial Data Exploration
4. Data Cleaning and Preprocessing
5. Exploratory Data Analysis (EDA)
6. Developing Key Metrics for Dashboard